<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day_19_EY_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Overwriting .env


In [2]:
!pip install -q langchain langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu azure-search-documents azure-identity python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
import os

for name in [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_OPENAI_EMBED_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION",
]:
    print(f"{name}: {os.getenv(name)}")

AZURE_OPENAI_ENDPOINT: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
AZURE_OPENAI_CHAT_DEPLOYMENT: gpt-4o-1
AZURE_OPENAI_EMBED_DEPLOYMENT: text-embedding-3-small
AZURE_OPENAI_API_VERSION: 2024-10-21


In [8]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ.get("AZURE_OPENAI_API_KEY")

print("Endpoint:", endpoint)
print("Embedding deployment:", os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"])

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.embeddings.create(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    input="hello"
)

print("Embedding dimension:", len(response.data[0].embedding))

Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
Embedding deployment: text-embedding-3-small
Embedding dimension: 1536


In [9]:
import os
from openai import AzureOpenAI
from langchain_openai import AzureOpenAIEmbeddings

def get_embeddings():
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]
    api_version = os.environ["AZURE_OPENAI_API_VERSION"]

    return AzureOpenAIEmbeddings(
        azure_endpoint=endpoint.split("/openai/")[0],
        api_key=api_key,
        azure_deployment=deployment,
        openai_api_version=api_version,
    )

In [10]:
emb = get_embeddings()

vector = emb.embed_query("hello")

print("Embedding dimension:", len(vector))
print("Azure OpenAI embeddings working!")

Embedding dimension: 1536
Azure OpenAI embeddings working!


In [11]:
emb = get_embeddings()
print("Embedding dimension:", len(emb.embed_query("hello")))

Embedding dimension: 1536


In [14]:
import os

print("Chat deployment:", os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"))
print("API version:", os.getenv("AZURE_OPENAI_API_VERSION"))

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
print("Endpoint:", endpoint)

Chat deployment: gpt-4o-1
API version: 2024-10-21
Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1


In [15]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "user", "content": "Reply with the single word: OK"}
    ],
    max_tokens=10
)

print("Chat deployment response:", response.choices[0].message.content)

Chat deployment response: OK


In [18]:
# Day 19 - Lab 1: Create the PolicyPilot document corpus

documents = [
    {
        "id": "HR-001",
        "title": "Work From Home Policy",
        "content": """
        Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.
        """
    },
    {
        "id": "TRV-002",
        "title": "Business Travel Policy",
        "content": """
        Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.
        """
    },
    {
        "id": "SEC-003",
        "title": "Information Security Policy",
        "content": """
        Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.
        """
    },
    {
        "id": "CDP-004",
        "title": "Learning and Development Policy",
        "content": """
        Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.
        """
    },
    {
        "id": "PRC-005",
        "title": "Procurement Policy",
        "content": """
        Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.
        """
    }
]

print("Number of policy documents:", len(documents))

for doc in documents:
    print(doc["id"], "-", doc["title"])

Number of policy documents: 5
HR-001 - Work From Home Policy
TRV-002 - Business Travel Policy
SEC-003 - Information Security Policy
CDP-004 - Learning and Development Policy
PRC-005 - Procurement Policy


In [19]:
%pip install -q langchain-text-splitters

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["content"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "id": doc["id"],
            "title": doc["title"],
            "chunk_number": i + 1,
            "text": chunk
        })

print("Total chunks created:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks created: 5

First chunk:
{'id': 'HR-001', 'title': 'Work From Home Policy', 'chunk_number': 1, 'text': 'Employees may work from home up to two days per week with\n        manager approval. Employees must remain available during\n        core business hours from 10 AM to 4 PM. Confidential company\n        information must only be accessed through approved systems.'}


In [21]:
%pip install -q faiss-cpu langchain-community

In [22]:
from langchain_core.documents import Document

department_map = {
    "HR-001": "HR",
    "TRV-002": "Finance",
    "SEC-003": "IT",
    "CDP-004": "Legal",
    "PRC-005": "Finance"
}

docs = []

for chunk in chunks:
    docs.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk["id"],
                "title": chunk["title"],
                "department": department_map[chunk["id"]],
                "chunk_id": f'{chunk["id"]}-{chunk["chunk_number"] - 1}'
            }
        )
    )

print("Documents prepared:", len(docs))
print("Sample metadata:", docs[0].metadata)

Documents prepared: 5
Sample metadata: {'source': 'HR-001', 'title': 'Work From Home Policy', 'department': 'HR', 'chunk_id': 'HR-001-0'}


In [25]:
from importlib.metadata import version

print("langchain-openai:", version("langchain-openai"))
print("openai:", version("openai"))

langchain-openai: 1.7.3
openai: 2.54.0


In [26]:
import os
from langchain_openai import OpenAIEmbeddings

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")

emb = OpenAIEmbeddings(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    base_url=endpoint + "/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

vector = emb.embed_query("Test Azure embeddings")

print("Embedding dimension:", len(vector))
print("Azure v1 embeddings working!")

Embedding dimension: 1536
Azure v1 embeddings working!


In [27]:
from langchain_community.vectorstores import FAISS

# Use the working Azure v1 embedding client
faiss_store = FAISS.from_documents(docs, emb)

print("FAISS vector database created successfully!")
print("Total policy chunks indexed:", len(docs))

FAISS vector database created successfully!
Total policy chunks indexed: 5


In [28]:
query = "What approval is required before booking business travel?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: TRV-002
Title: Business Travel Policy
Content: Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.

Result 2
Source: PRC-005
Title: Procurement Policy
Content: Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


In [29]:
query = "What should employees do if they receive a phishing email?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Result 2
Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


In [30]:
%pip install -q azure-search-documents

In [32]:
from google.colab import userdata
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)

endpoint = userdata.get("AZURE_SEARCH_ENDPOINT")
key = userdata.get("AZURE_SEARCH_KEY")

index_name = "policypilot-index"

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchableField(
        name="title",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="department",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(
            SearchFieldDataType.Single
        ),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="policy-vector-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="policy-hnsw")
    ],
    profiles=[
        VectorSearchProfile(
            name="policy-vector-profile",
            algorithm_configuration_name="policy-hnsw"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

index_client = SearchIndexClient(
    endpoint=endpoint,
    credential=__import__(
        "azure.core.credentials", fromlist=["AzureKeyCredential"]
    ).AzureKeyCredential(key)
)

result = index_client.create_or_update_index(index)

print("Index created successfully:", result.name)

Index created successfully: policypilot-index


In [34]:
existing_index = index_client.get_index("policypilot-index")

print("Existing index:", existing_index.name)
print("\nExisting fields:")

for field in existing_index.fields:
    print(
        f"Name: {field.name}, "
        f"Type: {field.type}, "
        f"Key: {field.key}, "
        f"Searchable: {field.searchable}, "
        f"Filterable: {field.filterable}"
    )

Existing index: policypilot-index

Existing fields:
Name: id, Type: SearchFieldDataType.STRING, Key: True, Searchable: False, Filterable: False
Name: content, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: source, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: title, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: department, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: content_vector, Type: Collection(Edm.Single), Key: False, Searchable: True, Filterable: False


In [37]:
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True)

required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_EMBED_DEPLOYMENT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_SEARCH_KEY",
]

missing = [name for name in required_vars if not os.getenv(name)]

if missing:
    print("Missing configuration:", missing)
    print("Please load these values from Colab Secrets.")
else:
    print("Configuration check passed.")
    print("Endpoint:", os.environ["AZURE_OPENAI_ENDPOINT"])
    print("Embedding deployment:", os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"])
    print("Chat deployment:", os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"])

Configuration check passed.
Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
Embedding deployment: text-embedding-3-small
Chat deployment: gpt-4o-1


In [38]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.embeddings.create(
    model=deployment,
    input="Test Azure embeddings"
)

print("Embedding test successful")
print("Embedding dimension:", len(response.data[0].embedding))

Embedding test successful
Embedding dimension: 1536


In [39]:
print("Number of policy chunks:", len(docs))
print("\nFirst policy chunk:")
print("Content:", docs[0].page_content)
print("Metadata:", docs[0].metadata)

Number of policy chunks: 5

First policy chunk:
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.
Metadata: {'source': 'HR-001', 'title': 'Work From Home Policy', 'department': 'HR', 'chunk_id': 'HR-001-0'}


In [40]:
import os
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

# 1. Connect to Azure OpenAI v1 endpoint
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=endpoint + "/"
)

embedding_deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]

# 2. Connect to the existing Azure AI Search index
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# 3. Generate embeddings and prepare search documents
search_documents = []

for i, doc in enumerate(docs):
    embedding_response = openai_client.embeddings.create(
        model=embedding_deployment,
        input=doc.page_content
    )

    vector = embedding_response.data[0].embedding
    metadata = doc.metadata

    search_documents.append({
        "id": str(i + 1),
        "content": doc.page_content,
        "source": str(metadata.get("source", "unknown")),
        "title": str(metadata.get("title", "Untitled")),
        "department": str(metadata.get("department", "General")),
        "content_vector": vector
    })

print("Prepared documents:", len(search_documents))
print("Embedding dimensions:",
      len(search_documents[0]["content_vector"]))

# 4. Upload documents to the existing index
results = search_client.upload_documents(
    documents=search_documents
)

successful = sum(1 for result in results if result.succeeded)
failed = len(results) - successful

print("Successfully uploaded:", successful)
print("Failed uploads:", failed)

for result in results:
    if not result.succeeded:
        print("Failed document:", result.key, result.error_message)

Prepared documents: 5
Embedding dimensions: 1536
Successfully uploaded: 5
Failed uploads: 0


In [41]:
print("Documents currently in the index:")

for result in search_client.search(
    search_text="phishing security policy",
    select=["id", "title", "source", "department", "content"],
    top=3
):
    print("\nTitle:", result.get("title"))
    print("Source:", result.get("source"))
    print("Department:", result.get("department"))
    print("Content:", result.get("content"))

Documents currently in the index:

Title: Information Security Policy
Source: SEC-003
Department: IT
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Title: Work From Home Policy
Source: HR-001
Department: HR
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Title: Procurement Policy
Source: PRC-005
Department: Finance
Content: Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.


In [42]:
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
import os

# Azure OpenAI client
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/"
)

# Azure AI Search client
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

print("Azure OpenAI and Azure AI Search clients are ready.")

Azure OpenAI and Azure AI Search clients are ready.


In [43]:
query = "What should employees do if they receive a phishing email?"

# Convert the question into an embedding
embedding_response = openai_client.embeddings.create(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    input=query
)

query_vector = embedding_response.data[0].embedding

# Search using vector similarity
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=3,
    fields="content_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["id", "title", "source", "department", "content"],
    top=3
)

for rank, result in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Title:", result.get("title"))
    print("Source:", result.get("source"))
    print("Department:", result.get("department"))
    print("Content:", result.get("content"))


Result 1
Title: Information Security Policy
Source: SEC-003
Department: IT
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Result 2
Title: Work From Home Policy
Source: HR-001
Department: HR
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Result 3
Title: Learning and Development Policy
Source: CDP-004
Department: Legal
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


In [44]:
test_questions = [
    "How many days do employees have to submit travel expenses?",
    "Can employees work from home without manager approval?"
]

for query in test_questions:
    print("\n" + "=" * 60)
    print("Question:", query)

    response = openai_client.embeddings.create(
        model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
        input=query
    )

    vector_query = VectorizedQuery(
        vector=response.data[0].embedding,
        k_nearest_neighbors=3,
        fields="content_vector"
    )

    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["title", "source", "content"],
        top=3
    )

    for result in results:
        print("\nPolicy:", result.get("title"))
        print("Source:", result.get("source"))
        print("Content:", result.get("content"))


Question: How many days do employees have to submit travel expenses?

Policy: Business Travel Policy
Source: TRV-002
Content: Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.

Policy: Work From Home Policy
Source: HR-001
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Policy: Learning and Development Policy
Source: CDP-004
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with thei

In [45]:
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
import os

# Azure OpenAI client
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/"
)

# Azure AI Search client
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]
CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]


def ask_policy_assistant(question):
    # 1. Convert the user's question into an embedding
    embedding_response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=question
    )
    query_vector = embedding_response.data[0].embedding

    # 2. Retrieve relevant policy chunks
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )

    search_results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["title", "source", "department", "content"],
        top=3
    )

    retrieved_chunks = list(search_results)

    if not retrieved_chunks:
        return "No matching policy documents were found.", []

    # 3. Prepare the context for the language model
    context = "\n\n".join(
        f"Policy: {item.get('title', 'Unknown')}\n"
        f"Source: {item.get('source', 'Unknown')}\n"
        f"Content: {item.get('content', '')}"
        for item in retrieved_chunks
    )

    # 4. Ask Azure OpenAI to answer using the retrieved context
    completion = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a company policy assistant. Answer using only "
                    "the supplied policy context. If the context does not "
                    "contain the answer, say that the policy information "
                    "provided is insufficient. Do not invent policy rules."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Policy context:\n{context}\n\n"
                    f"Employee question: {question}"
                )
            }
        ],
        temperature=0
    )

    answer = completion.choices[0].message.content
    return answer, retrieved_chunks

In [46]:
question = "What should employees do if they receive a phishing email?"

answer, sources = ask_policy_assistant(question)

print("QUESTION:")
print(question)

print("\nRAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for source in sources:
    print(
        f"- {source.get('title')} "
        f"({source.get('source')}, "
        f"Department: {source.get('department')})"
    )

QUESTION:
What should employees do if they receive a phishing email?

RAG ANSWER:
The policy information provided states that suspected phishing emails must be reported immediately to the IT security team.

RETRIEVED SOURCES:
- Information Security Policy (SEC-003, Department: IT)
- Work From Home Policy (HR-001, Department: HR)
- Learning and Development Policy (CDP-004, Department: Legal)


In [47]:
questions = [
    "How many days do employees have to submit travel expenses?",
    "Can employees work from home without manager approval?",
    "What authentication method must employees use?",
    "What is the policy for purchasing items for the company?"
]

for question in questions:
    answer, sources = ask_policy_assistant(question)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", answer)
    print("SOURCES:", ", ".join(
        str(source.get("title", "Unknown")) for source in sources
    ))


QUESTION: How many days do employees have to submit travel expenses?
ANSWER: Employees must submit travel expenses with receipts within seven working days after completing the trip, as per the Business Travel Policy (TRV-002).
SOURCES: Business Travel Policy, Work From Home Policy, Learning and Development Policy

QUESTION: Can employees work from home without manager approval?
ANSWER: The policy information provided is insufficient. The Work From Home Policy (HR-001) states that employees may work from home up to two days per week with manager approval, but it does not specify whether working from home without manager approval is allowed or under what circumstances.
SOURCES: Work From Home Policy, Learning and Development Policy, Business Travel Policy

QUESTION: What authentication method must employees use?
ANSWER: The policy information provided specifies that employees must use multifactor authentication for company systems, as stated in the Information Security Policy (Source: S

In [48]:

# Day 19: RAG Answer Quality Evaluation

evaluation_questions = [
    {
        "question": "What should employees do if they receive a phishing email?",
        "expected": "The answer should be supported by the Information Security Policy.",
        "type": "Answerable"
    },
    {
        "question": "What approval is required before booking business travel?",
        "expected": "The answer should be supported by the Business Travel Policy.",
        "type": "Answerable"
    },
    {
        "question": "What is the company's annual bonus percentage?",
        "expected": "If the policy corpus does not specify this, say so.",
        "type": "Out of scope"
    },
    {
        "question": "What is the CEO's personal phone number?",
        "expected": "Do not invent or disclose a number from unsupported sources.",
        "type": "Out of scope"
    },
    {
        "question": "How many days of paid parental leave are provided?",
        "expected": "Answer only if the retrieved policy explicitly specifies it.",
        "type": "Check coverage"
    }
]

for item in evaluation_questions:
    print("\n" + "=" * 75)
    print("TEST TYPE:", item["type"])
    print("QUESTION:", item["question"])
    print("EXPECTED:", item["expected"])

    retrieved_docs = faiss_store.similarity_search(
        item["question"], k=3
    )

    print("\nRETRIEVED POLICY CHUNKS:")
    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\n{rank}. Source: {doc.metadata.get('source')}")
        print("Title:", doc.metadata.get("title"))
        print("Content:", doc.page_content[:700])



TEST TYPE: Answerable
QUESTION: What should employees do if they receive a phishing email?
EXPECTED: The answer should be supported by the Information Security Policy.

RETRIEVED POLICY CHUNKS:

1. Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

2. Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

3. Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Em

In [49]:

# Day 19: RAG Answer Quality Evaluation

evaluation_questions = [
    {
        "question": "What should employees do if they receive a phishing email?",
        "expected": "The answer should be supported by the Information Security Policy.",
        "type": "Answerable"
    },
    {
        "question": "What approval is required before booking business travel?",
        "expected": "The answer should be supported by the Business Travel Policy.",
        "type": "Answerable"
    },
    {
        "question": "What is the company's annual bonus percentage?",
        "expected": "If the policy corpus does not specify this, say so.",
        "type": "Out of scope"
    },
    {
        "question": "What is the CEO's personal phone number?",
        "expected": "Do not invent or disclose a number from unsupported sources.",
        "type": "Out of scope"
    },
    {
        "question": "How many days of paid parental leave are provided?",
        "expected": "Answer only if the retrieved policy explicitly specifies it.",
        "type": "Check coverage"
    }
]

for item in evaluation_questions:
    print("\n" + "=" * 75)
    print("TEST TYPE:", item["type"])
    print("QUESTION:", item["question"])
    print("EXPECTED:", item["expected"])

    retrieved_docs = faiss_store.similarity_search(
        item["question"], k=3
    )

    print("\nRETRIEVED POLICY CHUNKS:")
    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\n{rank}. Source: {doc.metadata.get('source')}")
        print("Title:", doc.metadata.get("title"))
        print("Content:", doc.page_content[:700])



TEST TYPE: Answerable
QUESTION: What should employees do if they receive a phishing email?
EXPECTED: The answer should be supported by the Information Security Policy.

RETRIEVED POLICY CHUNKS:

1. Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

2. Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

3. Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Em

In [50]:

%pip install -q pinecone langchain-pinecone


In [51]:

import os
from google.colab import userdata

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "policypilot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(emb.embed_query("probe")),
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Pinecone index is ready:", index_name)


Pinecone index is ready: policypilot


In [52]:

pine_store = PineconeVectorStore.from_documents(
    docs,
    embedding=emb,
    index_name="policypilot"
)

test_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?"
]

for question in test_questions:
    print("\nQUESTION:", question)

    results = pine_store.similarity_search(question, k=3)

    for rank, doc in enumerate(results, start=1):
        print(
            f"{rank}. Source: {doc.metadata.get('source')} | "
            f"Policy: {doc.metadata.get('title')}"
        )
        print(doc.page_content[:400])



QUESTION: What should I do if my work laptop is stolen?
1. Source: SEC-003 | Policy: Information Security Policy
3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.
 
4. Cloud storage. Client or company data must never be stored on personal cloud accounts such as personal Google Drive or Dropbox.
2. Source: SEC-003 | Policy: Information Security Policy
Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corporate applications and VPN access.
 
2. Devices. All laptops are encrypted by default. USB storage devices are disabled by default; exceptions require written approval from the IT Security team.
3. Source: SEC-003 | Policy: Information Security Policy
Employe

In [53]:

comparison_questions = [
    "What should I do if my work laptop is stolen?",
    "Can I book business class for an eight-hour flight?",
    "How long are client engagement records kept?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for question in comparison_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    print("\nFAISS:")
    for doc in faiss_store.similarity_search(question, k=2):
        print(doc.metadata.get("source"), "|", doc.page_content[:200])

    print("\nPINECONE:")
    for doc in pine_store.similarity_search(question, k=2):
        print(doc.metadata.get("source"), "|", doc.page_content[:200])



QUESTION: What should I do if my work laptop is stolen?

FAISS:
SEC-003 | Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
       
HR-001 | Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        

PINECONE:
SEC-003 | 3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not a
SEC-003 | Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corpor

QUESTION: Can I book business class for an eight-hour flight?

FAISS:
TRV-002 | 

In [54]:

# Verify that Pinecone contains and retrieves policy chunks

test_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for question in test_questions:
    print("\n" + "=" * 75)
    print("QUESTION:", question)

    results = pine_store.similarity_search(question, k=3)

    if not results:
        print("No matching policy chunks retrieved.")
        continue

    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Source:", doc.metadata.get("source", "Not available"))
        print("Title:", doc.metadata.get("title", "Not available"))
        print("Policy text:", doc.page_content[:500])



QUESTION: What should I do if my work laptop is stolen?

Result 1
Source: SEC-003
Title: Information Security Policy
Policy text: 3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.
 
4. Cloud storage. Client or company data must never be stored on personal cloud accounts such as personal Google Drive or Dropbox.

Result 2
Source: SEC-003
Title: Information Security Policy
Policy text: Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corporate applications and VPN access.
 
2. Devices. All laptops are encrypted by default. USB storage devices are disabled by default; exceptions require written approval from the IT Security team.

Result 3
Source: SEC-003
Titl

In [56]:
%pip install -q python-dotenv openai

In [57]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

# Load variables from your existing .env file
load_dotenv(".env", override=False)

# Read Azure OpenAI settings from .env
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# Find the deployment name using common .env variable names
chat_deployment = (
    os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
)

# Validate configuration without displaying secrets
missing = [
    name for name, value in {
        "AZURE_OPENAI_API_KEY": api_key,
        "AZURE_OPENAI_ENDPOINT": endpoint,
        "AZURE_OPENAI_API_VERSION": api_version,
        "Azure OpenAI deployment name": chat_deployment,
    }.items()
    if not value
]

if missing:
    print("Missing configuration:", ", ".join(missing))
    print("Check your .env variable names and file location.")
else:
    chat_client = AzureOpenAI(
        api_key=api_key,
        azure_endpoint=endpoint,
        api_version=api_version
    )
    print("Azure OpenAI client initialized successfully.")
    print("Deployment configuration found.")

Azure OpenAI client initialized successfully.
Deployment configuration found.


In [58]:
import os

print("Current directory:", os.getcwd())
print(".env exists:", os.path.isfile(".env"))

Current directory: /content
.env exists: True


In [61]:

%pip install -q python-dotenv openai

import os
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# 1. Load your existing .env file
# ==========================================
load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# Validate required settings
missing = [
    name for name, value in {
        "AZURE_OPENAI_ENDPOINT": endpoint,
        "AZURE_OPENAI_API_KEY": api_key,
        "AZURE_OPENAI_CHAT_DEPLOYMENT": chat_deployment,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        "Missing .env variables: " + ", ".join(missing)
    )

# ==========================================
# 2. Configure Azure OpenAI v1 client
# ==========================================
if endpoint.endswith("/openai/v1"):
    base_url = endpoint + "/"
elif endpoint.endswith("/openai"):
    base_url = endpoint + "/v1/"
else:
    base_url = endpoint + "/openai/v1/"

chat_client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

print("Azure OpenAI v1 client initialized.")
print("Chat deployment:", chat_deployment)
print("API version setting found:", bool(api_version))

# ==========================================
# 3. PolicyPilot RAG answer function
# Requires pine_store from your Pinecone setup
# ==========================================
def answer_from_pinecone(question, k=3):
    retrieved_docs = pine_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Policy: {doc.metadata.get('title', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = f"""
Answer using only the supplied policy excerpts.

Rules:
- Do not invent policy rules or facts.
- If there is insufficient evidence, say:
  "I couldn't find enough information in the available policies
  to answer this question."
- Cite the supporting policy source when possible.
- Related text alone does not prove that a question is answerable.

POLICY EXCERPTS:
{context if context else "No policy excerpts retrieved."}

QUESTION:
{question}

Give a concise answer, cite the supporting source,
or explain that the information was not found.
"""

    response = chat_client.chat.completions.create(
        model=chat_deployment,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are PolicyPilot, an evidence-based "
                    "policy assistant."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


# ==========================================
# 4. Test the RAG pipeline
# Requires test_questions from your earlier cell
# ==========================================
for question in test_questions:
    print("\n" + "=" * 75)
    print("QUESTION:", question)

    try:
        print(answer_from_pinecone(question))
    except Exception as e:
        print("Error type:", type(e).__name__)
        print("Error details:", str(e))
        break


Azure OpenAI v1 client initialized.
Chat deployment: gpt-4o-1
API version setting found: True

QUESTION: What should I do if my work laptop is stolen?
Report the stolen laptop to the Security Operations Center (SOC) within 1 hour of discovery using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.  
(Source: SEC-003, Information Security Policy, Section 3)

QUESTION: What is the hotel limit for a Tier-1 city?
The hotel limit for a Tier-1 city is 180 USD per night.  
(Source: TRV-002, Travel & Expense Policy)

QUESTION: How long are client engagement records retained?
Client engagement records are retained for 7 years after engagement closure. (Source: CDP-004, Section 4)

QUESTION: What is the company's annual bonus percentage?
"I couldn't find enough information in the available policies to answer this question."

QUESTION: What is the company's policy on cryptocurrency trading?
"I couldn't find enough information in the available policies 

In [62]:

evaluation_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for i, question in enumerate(evaluation_questions, start=1):
    print(f"\n{'=' * 75}")
    print(f"TEST CASE {i}: {question}")

    retrieved_docs = pine_store.similarity_search(question, k=3)

    print("\nRetrieved sources:")
    for doc in retrieved_docs:
        print("-", doc.metadata.get("source", "Unknown"))

    print("\nPolicyPilot answer:")
    print(answer_from_pinecone(question))



TEST CASE 1: What should I do if my work laptop is stolen?

Retrieved sources:
- SEC-003
- SEC-003
- SEC-003

PolicyPilot answer:
Report the stolen laptop to the Security Operations Center (SOC) within 1 hour of discovery using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.  
(Source: SEC-003, Information Security Policy, Section 3)

TEST CASE 2: What is the hotel limit for a Tier-1 city?

Retrieved sources:
- TRV-002
- TRV-002
- TRV-002

PolicyPilot answer:
The hotel limit for a Tier-1 city is 180 USD per night. (Source: TRV-002, Travel & Expense Policy, Section 2)

TEST CASE 3: How long are client engagement records retained?

Retrieved sources:
- CDP-004
- CDP-004
- CDP-004

PolicyPilot answer:
Client engagement records are retained for 7 years after engagement closure. (Source: CDP-004, Section 4)

TEST CASE 4: What is the company's annual bonus percentage?

Retrieved sources:
- HR-001
- CDP-004
- SEC-003

PolicyPilot answer:
"I coul